In [1]:
import pandas as pd
import numpy as np
import wfdb
import ast

In [2]:
dataset_directory = "./data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

In [3]:
database_path = dataset_directory + "/ptbxl_database.csv"
database = pd.read_csv(database_path, index_col="ecg_id")

In [4]:
database.head()

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [5]:
database.scp_codes = database.scp_codes.apply(lambda x: ast.literal_eval(x))

In [6]:
database.scp_codes

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21837, dtype: object

In [10]:
database.scp_codes.isna().sum()

np.int64(0)

In [15]:
database.scp_codes

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21837, dtype: object

In [17]:
database.scp_codes.iloc[21832]

{'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'STACH': 0.0}

In [19]:
scp_statements_path = dataset_directory + "/scp_statements.csv"
scp_statements = pd.read_csv(scp_statements_path, index_col=0)

In [20]:
scp_statements.head()

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [200]:
scp_statements.diagnostic_subclass.unique()

array(['STTC', 'NST_', 'NORM', 'IMI', 'AMI', 'LVH', 'LAFB/LPFB', 'ISC_',
       'IRBBB', '_AVB', 'IVCD', 'ISCA', 'CRBBB', 'CLBBB', 'LAO/LAE',
       'ISCI', 'LMI', 'RVH', 'RAO/RAE', 'WPW', 'ILBBB', 'SEHYP', 'PMI',
       nan], dtype=object)

In [201]:
len(database.index)

21837

In [216]:
superclasses = {code:np.zeros(shape=len(database.index), dtype=np.uint8) for code in scp_statements.diagnostic_class.dropna().unique()}

In [217]:
len(superclasses)

5

In [218]:
superclasses

{'STTC': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'NORM': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'MI': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'HYP': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'CD': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8)}

In [219]:
subclasses = {subcode:np.zeros(shape=len(database.index), dtype=np.uint8) for subcode in scp_statements.diagnostic_subclass.dropna().unique()}

In [220]:
subclasses

{'STTC': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'NST_': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'NORM': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'IMI': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'AMI': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'LVH': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'LAFB/LPFB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'ISC_': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'IRBBB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 '_AVB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'IVCD': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'ISCA': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'CRBBB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'CLBBB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'LAO/LAE': arr

In [221]:
super_aggregate = {code:[] for code in scp_statements.diagnostic_class.dropna().unique()}

In [222]:
super_aggregate

{'STTC': [], 'NORM': [], 'MI': [], 'HYP': [], 'CD': []}

In [223]:
def append_to_class(class_dict, assigned_keys):
    class_keys = list(class_dict.keys())
    print(class_keys)

    
    for assigned in assigned_keys:
        class_keys.find(assigned)
        assert False
        class_dict[assigned].append(1)

    for k in class_keys:
        class_dict[k].append(0)
    
    # for k, val in class_dict.items(): 
    #     for assigned in assigned_keys:
    #         if k == assigned :
    #             val.append(1)
        
                

In [225]:
def aggregate_into_super_and_sub_classes(database, scp_codes_database, superclasses, subclasses, super_aggregate):

    for i, scp in enumerate(database.scp_codes):
        
        superclass_keys = list()
        subclass_keys = list()
        for key, val in scp.items():
            # print(key, val)
            if val > 0:
                entry = scp_statements.loc[key]
                superclass_keys.append(entry.diagnostic_class)
                subclass_keys.append(entry.diagnostic_subclass)
                # print(key, superclass_keys, subclass_keys)
        
        # print(superclass_keys)
        # print(subclass_keys)
        
        for key, val in superclasses.items():
            for k in superclass_keys: 
                if key == k:
                    val[i] = 1

        for key, val in subclasses.items():
            for k in subclass_keys: 
                if key == k:
                    val[i] = 1
        # append_to_class(superclasses, superclass_keys)
        # append_to_class(subclasses, subclass_keys)

            # super_aggregate[superclass]
            
        # break
    # print(i, scp)

In [226]:
aggregate_into_super_and_sub_classes(database, scp_statements, superclasses, subclasses, super_aggregate)

In [227]:
superclasses

{'STTC': array([0, 0, 0, ..., 1, 0, 0], shape=(21837,), dtype=uint8),
 'NORM': array([1, 1, 1, ..., 0, 1, 1], shape=(21837,), dtype=uint8),
 'MI': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'HYP': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'CD': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8)}

In [228]:
subclasses

{'STTC': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'NST_': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'NORM': array([1, 1, 1, ..., 0, 1, 1], shape=(21837,), dtype=uint8),
 'IMI': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'AMI': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'LVH': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'LAFB/LPFB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'ISC_': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'IRBBB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 '_AVB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'IVCD': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'ISCA': array([0, 0, 0, ..., 1, 0, 0], shape=(21837,), dtype=uint8),
 'CRBBB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'CLBBB': array([0, 0, 0, ..., 0, 0, 0], shape=(21837,), dtype=uint8),
 'LAO/LAE': arr

In [229]:
np.count_nonzero(subclasses["PMI"])

17

In [230]:
scp_statements.loc["NDT"].diagnostic_class

'STTC'

In [231]:
database

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,CLBBB,LAO/LAE,ISCI,LMI,RVH,RAO/RAE,WPW,ILBBB,SEHYP,PMI
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,0,0,0,0,0,0,0,0,0,0
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,0,0,0,0,0,0,0,0,0,0
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,0,0,0,0,0,0,0,0,0,0
21834,20703.0,93.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,0,0,0,0,0,0,0,0,0,0
21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,0,0,0,0,0,0,0,0,0,0


In [232]:
for key, val in superclasses.items():
    database[key] = val

In [233]:
for key, val in subclasses.items():
    database[key] = val

In [235]:
database.head()[["patient_id", "scp_codes", *list(superclasses.keys())]] # , *list(subclasses.keys())]]

,patient_id,scp_codes,STTC,NORM,MI,HYP,CD
ecg_id,,,,,,,
1,15709.0,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",0,1,0,0,0
2,13243.0,"{'NORM': 80.0, 'SBRAD': 0.0}",0,1,0,0,0
3,20372.0,"{'NORM': 100.0, 'SR': 0.0}",0,1,0,0,0
4,17014.0,"{'NORM': 100.0, 'SR': 0.0}",0,1,0,0,0
5,17448.0,"{'NORM': 100.0, 'SR': 0.0}",0,1,0,0,0


In [280]:
classes_per_line = list()
multilabel = 0
no_label = 0
to_drop = list()

sup_classes = superclasses.keys()
for i in range(len(database)):
    c = list()
    for s in sup_classes:
        # print(s)
        # print(s, database[s].iloc[i])
        if database[s].iloc[i] == 1: c.append(s)
    classes_per_line.append(c)
    if len(c) > 1: 
        multilabel += 1
        to_drop.append(1)
    elif len(c) == 0: 
        no_label += 1
        to_drop.append(1)
    else:
        to_drop.append(0)
    # break

In [274]:
c

['NORM']

In [275]:
classes_per_line

[['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['MI'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 [],
 [],
 ['NORM'],
 [],
 ['NORM'],
 ['STTC'],
 [],
 ['NORM'],
 ['NORM'],
 ['STTC'],
 ['NORM'],
 ['STTC'],
 ['NORM'],
 ['HYP'],
 ['NORM'],
 ['CD'],
 ['NORM'],
 [],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['STTC', 'MI'],
 ['NORM'],
 ['CD'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['HYP', 'CD'],
 ['NORM'],
 ['NORM'],
 ['STTC'],
 ['CD'],
 ['MI', 'CD'],
 ['NORM'],
 ['CD'],
 ['NORM'],
 ['STTC'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['MI'],
 ['NORM'],
 ['CD'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['MI', 'CD'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['STTC', 'CD'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'],
 ['NORM'

In [307]:
classes_per_line = np.array(classes_per_line)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (21837,) + inhomogeneous part.

In [278]:
multilabel

3877

In [279]:
no_label

1325

In [286]:
to_drop

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [291]:
np.array(to_drop).astype(bool)

array([False, False, False, ...,  True, False, False], shape=(21837,))

In [282]:
np.sum(to_drop)

np.int64(5202)

In [283]:
multilabel+no_label

5202

In [261]:
database["NORM"].iloc[0]

np.uint8(1)

In [299]:
database[~np.array(to_drop).astype(bool)]

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,CLBBB,LAO/LAE,ISCI,LMI,RVH,RAO/RAE,WPW,ILBBB,SEHYP,PMI
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,0,0,0,0,0,0,0,0,0,0
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,0,0,0,0,0,0,0,0,0,0
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21832,7954.0,63.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-05-30 14:14:25,sinusrhythmus linkstyp periphere niederspannun...,...,0,0,0,0,0,0,0,0,0,0
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,0,0,0,0,0,0,0,0,0,0
21834,20703.0,93.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,0,0,0,0,0,0,0,0,0,0


In [300]:
database

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,CLBBB,LAO/LAE,ISCI,LMI,RVH,RAO/RAE,WPW,ILBBB,SEHYP,PMI
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,0,0,0,0,0,0,0,0,0,0
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,0,0,0,0,0,0,0,0,0,0
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,0,0,0,0,0,0,0,0,0,0
21834,20703.0,93.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,0,0,0,0,0,0,0,0,0,0
21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,0,0,0,0,0,0,0,0,0,0


In [301]:
16635 - 21837

-5202

In [302]:
database.columns

Index(['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site',
       'device', 'recording_date', 'report', 'scp_codes', 'heart_axis',
       'infarction_stadium1', 'infarction_stadium2', 'validated_by',
       'second_opinion', 'initial_autogenerated_report', 'validated_by_human',
       'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems',
       'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr',
       'STTC', 'NORM', 'MI', 'HYP', 'CD', 'NST_', 'IMI', 'AMI', 'LVH',
       'LAFB/LPFB', 'ISC_', 'IRBBB', '_AVB', 'IVCD', 'ISCA', 'CRBBB', 'CLBBB',
       'LAO/LAE', 'ISCI', 'LMI', 'RVH', 'RAO/RAE', 'WPW', 'ILBBB', 'SEHYP',
       'PMI'],
      dtype='object')

In [306]:
database.filename_hr

ecg_id
1        records500/00000/00001_hr
2        records500/00000/00002_hr
3        records500/00000/00003_hr
4        records500/00000/00004_hr
5        records500/00000/00005_hr
                   ...            
21833    records500/21000/21833_hr
21834    records500/21000/21834_hr
21835    records500/21000/21835_hr
21836    records500/21000/21836_hr
21837    records500/21000/21837_hr
Name: filename_hr, Length: 21837, dtype: object

In [308]:
data = data = wfdb.rdsamp(dataset_directory+"/"+database.iloc[0].filename_hr)

In [309]:
data

(array([[-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
        ...,
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
        [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ]],
       shape=(5000, 12)),
 {'fs': 500,
  'sig_len': 5000,
  'n_sig': 12,
  'base_date': None,
  'base_time': None,
  'units': ['mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV',
   'mV'],
  'sig_name': ['I',
   'II',
   'III',
   'AVR',
   'AVL',
   'AVF',
   'V1',
   'V2',
   'V3',
   'V4',
   'V5',
   'V6'],
  'comments': []})

In [313]:
data[0]

array([[-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
       [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
       [-0.115, -0.05 ,  0.065, ..., -0.035, -0.035, -0.075],
       ...,
       [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
       [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ],
       [ 0.21 ,  0.205, -0.005, ...,  0.185,  0.17 ,  0.18 ]],
      shape=(5000, 12))

In [325]:
np.sum(database.strat_fold < 9) + np.sum(database.strat_fold == 9) + np.sum(database.strat_fold == 10)

np.int64(21837)

In [323]:
np.sum(database.strat_fold== 9)

np.int64(2193)

In [326]:
database[database.strat_fold == 10]

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,CLBBB,LAO/LAE,ISCI,LMI,RVH,RAO/RAE,WPW,ILBBB,SEHYP,PMI
ecg_id,,,,,,,,,,,,,,,,,,,,,
9,18792.0,55.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-12-08 09:44:43,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
38,17076.0,40.0,0,NaN,72.0,2.0,0.0,CS-12 E,1985-02-15 11:48:22,sinusrhythmus schwierig bestimmbare qrs-achse,...,0,0,0,0,0,0,0,0,0,0
40,19501.0,60.0,0,NaN,85.0,2.0,0.0,CS-12 E,1985-02-20 11:43:45,sinusrhythmus linkstyp sonst normales ekg,...,0,0,0,0,0,0,0,0,0,0
57,16063.0,26.0,0,NaN,93.0,2.0,0.0,CS-12 E,1985-06-06 11:32:43,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
59,19475.0,54.0,0,NaN,67.0,2.0,0.0,CS-12 E,1985-06-12 06:36:01,sinusrhythmus normales ekg,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21809,12931.0,69.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-02-18 12:36:54,sinusrhythmus linkstyp qrs(t) abnorm inferi...,...,0,0,0,0,0,0,0,0,0,0
21812,20789.0,67.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-02-21 13:34:15,supraventrikulÄre arrhythmie a-v block i p-ver...,...,0,0,0,0,0,0,0,0,0,0
21818,19204.0,84.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-03-03 12:09:05,sinusrhythmus linkstyp mÄssige amplitudenkrite...,...,0,0,0,0,0,0,0,0,0,0
